In [ ]:
import numpy as np

In [ ]:
# Canonical fiducial parameters from table 1

# Stellar parameters
M_star = 1.0           # Solar masses
a = 5.0                # AU
Mdot = 1.0             # Jupiter masses per year (M_J/yr)


# Opacity 
kappa_0 = 10.0         # cm^2/g
nu_0 = 1e14            # Hz
eta = 1.0              # opacity index


# Planet parameters
Mp = 1.0                # Jupiter masses
Rp = 1e10              # cm
Bp = 500.0             # Gauss, magnetic field strength



# constants
sigmaB = 5.67e-5      # erg/cm^2/K^4, Stefan-Boltzmann constant
G = 6.67e-8           # cm^3/g/s^2, gravitational constant
h = 6.63e-27          # erg s, Planck constant
c = 3.00e10           # cm/s, speed of light
k = 1.38e-16          # erg/K, Boltzmann constant

In [ ]:
# ---------------------------------------------
# The protoplanet structure (pre-requisites)
# -----------------------------------------------

Rc = (a / 3) * (Mp / (3 * M_star))**(1/3)   # Disk outer boundary
RH =  a*(Mp / (3 * M_star))**(1/3)    # Hills radius

RX = ((Bp**4 * Rp**12) / (G * Mp * Mdot**2))**(1/7)   # Disk inner boundary
RX = 3.8 * Rp


# Create radial grid
R = np.geomspace(RX, RH, 200)

# Define the fraction of infalling material that initially strikes the disk

up = Rp/Rc   # planet radius over centrifugal radius of the collapse flow




fd_dict = {
    "polar":      1 - 3*up/2,
    "quasipolar": 1 - up,
    "isotropic":  1 - up/2,
    "quasiequatorial": 1 - (4/(3*np.pi))*up**(3/2),
    "equatorial": 1 - (3/8)*up**2
}




# Compute the planet temperature 
Lp = (G * Mp * Mdot / Rp) * (1 - Rp**3 / (3 * RX**3)) * (1 - fd_dict[mode] * Rp / RX)

Tp = (Lp / (4 * np.pi * sigmaB * Rp**2))**0.25     # planet blackbody temperature


Ld = fd_dict[mode] * G * M * Mdot / (2 * RX)    # disk luminosity


# 2. Disk temperature profile

TX = (fd_dict[mode] * G * Mp * Mdot / (8 * np.pi * sigmaB * RX**3) * (1 - RX / Rc)**-1)**0.25
Td = TX * (R / RX)**(-0.75)



# 3. Planet Spectral Luminosity



# mu0 (initial polar angle) is a cubic function of mu and R

zeta = Rc / R # parameter


# assume face on viewing angle
psi = 0    # viewing angle
mu = np.cos(psi)

mu0 = np.roots([zeta, 0.0, (1 - zeta), -mu])
mu0 = mu0[np.isreal(mu0)].real
mu0 = mu0[(mu0 >= 0) & (mu0 <= 1)][0]   # the first value that fulfils the condition

fi_dict = {
    "polar":  3 * mu0**2,
    "quasipolar":  2 * mu0,
    "isotropic":  1,
    "quasiequatorial":  (4 / np.pi) * np.sqrt(1 - mu0),
    "equatorial":  (3 / 2) * (1 - mu0)
}


# the function i is the probability density of different mu values for each case



vr = -np.sqrt(G * Mp / R * (2 - zeta * (1 - mu0**2)))   # radial velocity of incoming material
rho_r_mu = (Mdot * fi_dict[mode]) / (4 * np.pi * R**2 * np.abs(vr)) * (1 + zeta * (3 * mu0**2 - 1))**-1


Ncol = np.trapz(rho_r_mu[(R >= Rp) & (R <= RH)], R[(R >= Rp) & (R <= RH)])* mu  # Integrate from Rp to RH, multiplied by cos(phi) a constant viewing angle

B_nu = (2 * h * nu**3 / c**2) / (np.exp(h * nu / (k * Tp)) - 1)  # Planck function
Lp_nu = 4 * np.pi**2 * Rp**2 * B_nu(Tp) * np.exp(-kappa_nu * Ncol)


# 4. Disk spectral luminosity

# phi : disk angle on x-y plane, independent of r and mu if viewing angle is face on










1. **`r_prime`**: 200 radii between \(R_X\) and \(R_C\).  
2. **`s0_array`**: For each \(r'\), solve \(r(s_0) = R_H\).  
3. **`s` grid**: 200 points between 0 and \(s_0(r')\).  
4. Compute \(\rho(r(s), \mu(s))\).  
5. Integrate over \(s \;\rightarrow\) gives one \(N_{\text{col}}(r')\) per starting radius.  

Now you have **`Ncol_array`** with 200 values — one for each \(r'\).

In [ ]:
# azimuthal grid for φ
phi_vals = np.linspace(0, 2*np.pi, 100)

# radial grid for the disk (RX → RC, not RH this time)
R_disk = np.geomspace(RX, Rc, 200)

# corresponding disk temperatures
Td_disk = TX * (R_disk / RX)**(-0.75)

# for each (r', phi), compute s0, then integrate rho along s to get Ncol_d
Ncol_d = np.zeros((len(R_disk), len(phi_vals)))

for i, rp in enumerate(R_disk):
    for j, phi in enumerate(phi_vals):
        # sample s along the ray
        s = np.linspace(0, RH, 200)
        r_s = np.sqrt(rp**2 + s**2 + 2*rp*s*np.sin(psi)*np.cos(phi))  # Eq. 14a
        mu_s = (s / r_s) * np.cos(psi)                               # Eq. 14b
        rho_vals = (Mdot * fi_dict[mode]) / (4*np.pi * r_s**2 * np.abs(vr[i])) \
                   * (1 + zeta[i]*(3*mu0**2 - 1))**-1
        mask = r_s <= RH
        Ncol_d[i, j] = np.trapz(rho_vals[mask], s[mask])

# Planck function evaluated at each Td(r')
Bnu_disk = (2*h*nu**3 / c**2) / (np.exp(h*nu/(k*Td_disk)) - 1)

# integrand r * Bν(Td) * exp(-κν Ncol_d) averaged over φ
integrand_r = []
for i, rp in enumerate(R_disk):
    vals_phi = rp * Bnu_disk[i] * np.exp(-kappa_nu * Ncol_d[i, :])
    phi_int = np.trapz(vals_phi, phi_vals)
    integrand_r.append(phi_int)

integrand_r = np.array(integrand_r)

# integrate over r
r_int = np.trapz(integrand_r, R_disk)

# final disk spectral luminosity (Eq. 16)
Ld_nu = 4 * np.pi * np.cos(psi) * r_int